# Guía de Análisis de Datos Climáticos (CAMELS-CL)
¡Hola! Este Notebook (o libreta) contiene todo lo necesario para procesar los datos de las estaciones meteorológicas en Chile y calcular las cardinalidades solicitadas para el informe de Álgebra.

## ⚠️ Precauciones y Requisitos Previos
1. **Ubicación de archivos:** Asegúrate de tener los archivos base de CAMELS-CL (`tmax_cr2met_day.csv` y `tmin_cr2met_day.csv`) en la **misma carpeta** donde se encuentra este Notebook o los scripts `.py`.
2. **Librería Pandas:** Todo el procesamiento matemático usa `pandas`. Si no la tienes instalada en tu entorno, ejecuta la celda de abajo antes de continuar.


In [ ]:
!pip install pandas

## Script 1: Filtrado y Consolidación (Universal)
Este script toma los datos crudos, calcula la temperatura promedio diaria, aísla el último quinquenio (5 años) con datos y limpia cualquier valor nulo. 
Se ha modificado para ser **universal**, por lo que al ejecutarlo te pedirá que ingreses el código de la estación.


In [ ]:
import pandas as pd
import os

print("=== SCRIPT 1: LIMPIEZA Y CONSOLIDACIÓN DE DATOS ===")
codigo_estacion = input("Por favor, ingresa el código de la estación meteorológica (ej. 10440000): ").strip()

archivo_tmax = 'tmax_cr2met_day.csv'
archivo_tmin = 'tmin_cr2met_day.csv'

# Para Jupyter, detectamos el directorio de trabajo actual
directorio_actual = os.getcwd()
ruta_tmax = os.path.join(directorio_actual, archivo_tmax)
ruta_tmin = os.path.join(directorio_actual, archivo_tmin)

try:
    print(f"Cargando datos de la estación {codigo_estacion}...")
    df_tmax = pd.read_csv(ruta_tmax, low_memory=False)
    df_tmin = pd.read_csv(ruta_tmin, low_memory=False)
    
    # Extraer columnas y renombrar
    df_tmax = df_tmax[['date', codigo_estacion]].rename(columns={codigo_estacion: 'T_max'})
    df_tmin = df_tmin[['date', codigo_estacion]].rename(columns={codigo_estacion: 'T_min'})
    
    # Unificar y calcular T_mean
    df_unificado = pd.merge(df_tmax, df_tmin, on='date')
    df_unificado['T_mean'] = (df_unificado['T_max'] + df_unificado['T_min']) / 2
    
    # Filtro de los últimos 5 años
    df_unificado['date'] = pd.to_datetime(df_unificado['date'])
    anio_maximo = df_unificado['date'].dt.year.max()
    anio_minimo = anio_maximo - 4 
    
    df_5_anios = df_unificado[df_unificado['date'].dt.year >= anio_minimo].copy()
    df_5_anios.dropna(inplace=True)
    
    archivo_salida = os.path.join(directorio_actual, f'datos_consolidados_{codigo_estacion}_5anios.csv')
    df_5_anios.to_csv(archivo_salida, index=False)
    
    print(f"✅ ¡Éxito! Datos filtrados desde {anio_minimo} hasta {anio_maximo}.")
    print(f"Archivo guardado como: {archivo_salida}\n")

except FileNotFoundError:
    print(f"❌ Error: No se encontraron los archivos base (tmax_cr2met_day.csv o tmin). Verifica que estén en {directorio_actual}")
except KeyError:
    print(f"❌ Error: El código de estación {codigo_estacion} no existe en la base de datos o está mal escrito.")



## Script 2: Análisis y Cardinalidades (Universal)
Una vez que el Script 1 haya generado tu archivo consolidado, ejecuta esta celda. Se encargará de aplicar los filtros lógicos para los conjuntos A, B y C, calcular las intersecciones y exportar la tabla resumen.


In [ ]:
import pandas as pd
import os

print("=== SCRIPT 2: CÁLCULO DE CARDINALIDADES ===")
codigo_estacion = input("Ingresa el código de la estación a analizar (ej. 10440000): ").strip()
zona_geografica = input("Ingresa la zona geográfica (Norte, Centro o Sur): ").strip()

directorio_actual = os.getcwd()
nombre_archivo = f'datos_consolidados_{codigo_estacion}_5anios.csv'
ruta_datos = os.path.join(directorio_actual, nombre_archivo)

try:
    df = pd.read_csv(ruta_datos)
    
    # Conjuntos Lógicos
    cond_A = df['T_max'] > 30
    cond_B = df['T_min'] < 10
    cond_C = (df['T_mean'] >= 18) & (df['T_mean'] <= 22)
    
    resultados = {
        "Estación": codigo_estacion,
        "Zona": zona_geografica,
        "A": cond_A.sum(),
        "B": cond_B.sum(),
        "C": cond_C.sum(),
        "AnB": (cond_A & cond_B).sum(),
        "AnC": (cond_A & cond_C).sum(),
        "BnC": (cond_B & cond_C).sum(),
        "AnBnC": (cond_A & cond_B & cond_C).sum()
    }
    
    print("\n" + "-" * 50)
    print(f" RESULTADOS: ESTACIÓN {codigo_estacion} ({zona_geografica.upper()}) ")
    print("-" * 50)
    for clave, valor in resultados.items():
        print(f"{clave:<10}: {valor}")
    print("-" * 50)
    
    # Guardar en CSV
    df_resultados = pd.DataFrame([resultados])
    nombre_salida = f'resultados_cardinalidad_{codigo_estacion}.csv'
    df_resultados.to_csv(os.path.join(directorio_actual, nombre_salida), index=False)
    print(f"✅ Tabla exportada exitosamente como: {nombre_salida}")

except FileNotFoundError:
    print(f"❌ Error: No se encontró el archivo '{nombre_archivo}'. Ejecuta el Script 1 primero para esta estación.")

